# Snow-Insulated Hybrid Stefan Model for Thaw Depth (ALT)
## Снеговой гибрид Стефана для расчета глубины сезонного протаивания (ALT)

This notebook covers the math, implementation, and calibration of the **Hybrid Stefan Model**, which integrates the insulating effect of winter snow accumulation into the classical heat conduction equations.

Этот блокнот охватывает математику, реализацию и калибровку **Гибридной модели Стефана**, которая интегрирует теплоизоляционный эффект зимнего снежного покрова в классические уравнения теплопроводности.

### 1. Physics and Snow Insulation / Физика процесса и снеговое утепление

Snow is an extremely effective thermal insulator due to its porous structure and low density. The winter snow layer acts as a barrier, preventing winter cold infiltration into the ground. A thick snow cover keeps the underlying permafrost warmer in winter, meaning it starts the summer thaw season with a higher base temperature, resulting in increased seasonal thawing ($ALT$).

Снег является высокоэффективным теплоизолятором благодаря своей пористой структуре и низкой плотности. Зимний снежный покров действует как экран, препятствуя переохлаждению грунта зимой. Плотный и высокий слой снега сохраняет мерзлоту теплой зимой, поэтому летний сезон оттаивания начинается при более высокой базовой температуре грунта, что в итоге приводит к увеличению глубины сезонного протаивания ($ALT$).

We model this insulation effect with a linear modifier $\beta$ multiplied by the winter maximum snow depth ($H_{snow}$):
Мы моделируем этот изоляционный эффект с помощью линейного множителя с коэффициентом чувствительности $\beta$, умноженным на максимальную высоту снежного покрова ($H_{snow}$):

$$ALT = E_{base} \cdot \sqrt{DDT} \cdot (1.0 + \beta \cdot H_{snow})$$

Where / Где:
- $E_{base}$: Base Stefan physics coefficient / Базовый коэффициент физики Стефана.
- $DDT$: Degree-days of thawing / Градусо-сутки таяния (°C·сут).
- $H_{snow}$: Winter snow depth / Высота снежного покрова (см).
- $\beta$: Sensitivity coefficient for snow insulation / Коэффициент снеговой чувствительности ($1/cm$).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

print("Libraries imported. / Импорт библиотек успешно выполнен.")

### 2. Python Implementation / Реализация на Python

In [ ]:
def stefan_hybrid(X, E_base=4.1866, beta=0.00155):
    """
    Computes hybrid ALT.
    X is a tuple of (DDT, H_snow).
    """
    ddt, h_snow = X
    return E_base * np.sqrt(ddt) * (1.0 + beta * h_snow)

# Sample prediction / Пример прогноза
ddt_sample = 2150
snow_sample = 38
alt_val = stefan_hybrid((ddt_sample, snow_sample))

print(f"For DDT = {ddt_sample} °C-days and Snow = {snow_sample} cm:")
print(f"Predicted Thaw Depth (ALT) = {alt_val:.2f} cm")
print(f"При DDT = {ddt_sample} °С-сут и высоте снега = {snow_sample} см:")
print(f"Расчетная глубина протаивания ALT = {alt_val:.2f} см")

### 3. Empirical Calibration / Калибровка коэффициентов

We will fit the two parameters $E_{base}$ and $\beta$ using historical multi-variate observations in Central Yakutia.

In [ ]:
# Historical inputs / Исторические данные
ddt_hist = np.array([1980, 2050, 2150, 2220, 2300, 2350])
snow_hist = np.array([28, 32, 38, 41, 45, 52])
alt_actual = np.array([193.5, 196.2, 205.6, 207.2, 211.5, 216.3])

# Perform fit / Многопараметрический фитинг
popt, pcov = curve_fit(stefan_hybrid, (ddt_hist, snow_hist), alt_actual, p0=[4.0, 0.001])
E_base_cal, beta_cal = popt

print(f"Calibrated E_base = {E_base_cal:.5f}")
print(f"Calibrated beta = {beta_cal:.5f}")
print(f"Откалиброванный E_base = {E_base_cal:.5f}")
print(f"Откалиброванная снеговая чувствительность beta = {beta_cal:.5f}")

### 4. Interactive Snow Impact Plot / Влияние снежного покрова на протаивание

In [ ]:
snow_range = np.linspace(10, 60, 100)
# For a fixed DDT of 2150 / Для фиксированного значения DDT = 2150
alt_vs_snow = stefan_hybrid((2150, snow_range), E_base=E_base_cal, beta=beta_cal)

plt.figure(figsize=(10, 5))
plt.plot(snow_range, alt_vs_snow, '-', color='#22d3ee', linewidth=2.5, label='Stefan Snow-Insulated Model')
plt.scatter(snow_hist, alt_actual, color='#ef4444', s=60, label='Historical Records / Данные наблюдений')
plt.title('Effect of Max Winter Snow Depth on Seasonal Soil Thawing (ALT)\nВлияние высоты снежного покрова на глубину сезонного таяния (ALT)')
plt.xlabel('Maximum Snow Depth / Высота снежного покрова (см)')
plt.ylabel('Active Layer Thickness (ALT) / Глубина протаивания (см)')
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()
plt.show()